# 01 – Data Exploration: UCI *Predict Students' Dropout and Academic Success*

**Purpose:** understand the original dataset before we design the database or generate any synthetic data.

**Source:** UCI Machine Learning Repository, dataset id 697 (Realinho, Machado, Baptista & Martins, 2021/2022).
Records come from a higher-education institution in Portugal (Polytechnic Institute of Portalegre), covering students enrolled between the 2008/09 and 2018/19 academic years.

> **Read-only rule:** this notebook only *reads* `data/raw/data.csv`. It never writes to it.
> SHA-256 of the original file: `3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae`

## 1. Load the original CSV
The file uses **semicolons** (`;`) as separators and begins with a UTF-8 BOM, so we pass `sep=';'` and `encoding='utf-8-sig'`.

In [1]:
from pathlib import Path
import hashlib
import pandas as pd

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# Works whether the notebook is run from the project root or from notebooks/
RAW = Path("data/raw/data.csv")
if not RAW.exists():
    RAW = Path("../data/raw/data.csv")

EXPECTED_SHA256 = "3ef126de5cefff26eb11fbb4237f1a1401cb64b488e2f1d598c23cedeb4c45ae"
actual = hashlib.sha256(RAW.read_bytes()).hexdigest()
print("File:", RAW.resolve())
print("Original file unchanged:", actual == EXPECTED_SHA256)

df = pd.read_csv(RAW, sep=";", encoding="utf-8-sig")
print("Rows:", df.shape[0], "| Columns:", df.shape[1])

File: C:\Users\Admin\Downloads\University-Institutional-Analytics-Dashboard\data\raw\data.csv
Original file unchanged: True
Rows: 4424 | Columns: 37


## 2. First five records

In [2]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,Mother's occupation,Father's occupation,Admission grade,Displaced,Educational special needs,Debtor,Tuition fees up to date,Gender,Scholarship holder,Age at enrollment,International,Curricular units 1st sem (credited),Curricular units 1st sem (enrolled),Curricular units 1st sem (evaluations),Curricular units 1st sem (approved),Curricular units 1st sem (grade),Curricular units 1st sem (without evaluations),Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,5,9,127.3,1,0,0,1,1,0,20,0,0,0,0,0,0.000000,0,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,3,3,142.5,1,0,0,0,1,0,19,0,0,6,6,6,14.000000,0,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,9,9,124.8,1,0,0,0,1,0,19,0,0,6,0,0,0.000000,0,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,5,3,119.6,1,0,0,1,0,0,20,0,0,6,8,6,13.428571,0,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,9,9,141.5,0,0,0,1,0,0,45,0,0,6,9,5,12.333333,0,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


## 3. Column names
`repr()` is used so hidden characters are visible. Note that **`'Daytime/evening attendance\t'` contains a trailing tab character** and **`Nacionality`** is misspelled in the source. We will fix these only in the *processed* copy, never in the raw file.

In [3]:
for i, c in enumerate(df.columns, 1):
    print(f"{i:2d}. {c!r}")

 1. 'Marital status'
 2. 'Application mode'
 3. 'Application order'
 4. 'Course'
 5. 'Daytime/evening attendance\t'
 6. 'Previous qualification'
 7. 'Previous qualification (grade)'
 8. 'Nacionality'
 9. "Mother's qualification"
10. "Father's qualification"
11. "Mother's occupation"
12. "Father's occupation"
13. 'Admission grade'
14. 'Displaced'
15. 'Educational special needs'
16. 'Debtor'
17. 'Tuition fees up to date'
18. 'Gender'
19. 'Scholarship holder'
20. 'Age at enrollment'
21. 'International'
22. 'Curricular units 1st sem (credited)'
23. 'Curricular units 1st sem (enrolled)'
24. 'Curricular units 1st sem (evaluations)'
25. 'Curricular units 1st sem (approved)'
26. 'Curricular units 1st sem (grade)'
27. 'Curricular units 1st sem (without evaluations)'
28. 'Curricular units 2nd sem (credited)'
29. 'Curricular units 2nd sem (enrolled)'
30. 'Curricular units 2nd sem (evaluations)'
31. 'Curricular units 2nd sem (approved)'
32. 'Curricular units 2nd sem (grade)'
33. 'Curricular units 

## 4. Missing values

In [4]:
missing = df.isna().sum()
print("Total missing cells:", int(missing.sum()))
missing[missing > 0]

Total missing cells: 0


Series([], dtype: int64)

**Hidden 'missing' data.** There are no empty cells, but some zeros mean "no data". For example, a semester grade of `0` for a student who enrolled in units usually means they were not evaluated or dropped out, not that they scored zero. We count those here:

In [5]:
g1 = "Curricular units 1st sem (grade)"
e1 = "Curricular units 1st sem (enrolled)"
g2 = "Curricular units 2nd sem (grade)"
e2 = "Curricular units 2nd sem (enrolled)"
print("1st-sem grade = 0 but enrolled in units:", int(((df[g1] == 0) & (df[e1] > 0)).sum()))
print("2nd-sem grade = 0 but enrolled in units:", int(((df[g2] == 0) & (df[e2] > 0)).sum()))
print("Enrolled in 0 units in BOTH semesters:  ", int(((df[e1] == 0) & (df[e2] == 0)).sum()))

1st-sem grade = 0 but enrolled in units: 538
2nd-sem grade = 0 but enrolled in units: 690
Enrolled in 0 units in BOTH semesters:   180


**What the grade column really measures.** If the grade were an average over *all* units, failing students would show grades between 0 and 10. The check below shows that grade > 0 happens exactly when at least one unit was passed, so the grade averages **passed units only**. Pass rate (approved ÷ enrolled) is therefore the better signal of poor performance.

In [6]:
for s, g, e in [("1st", g1, e1), ("2nd", g2, e2)]:
    a = df[f"Curricular units {s} sem (approved)"]
    print(f"{s} sem | grade>0 but nothing passed: {int(((df[g] > 0) & (a == 0)).sum())}"
          f" | passed something but grade=0: {int(((a > 0) & (df[g] == 0)).sum())}"
          f" | lowest non-zero grade: {df.loc[df[g] > 0, g].min()}")
w = "Curricular units 2nd sem (without evaluations)"
print("Rows where 2nd-sem 'without evaluations' > enrolled (source data quirk):", int((df[w] > df[e2]).sum()))

1st sem | grade>0 but nothing passed: 0 | passed something but grade=0: 0 | lowest non-zero grade: 9.8
2nd sem | grade>0 but nothing passed: 0 | passed something but grade=0: 0 | lowest non-zero grade: 10.0
Rows where 2nd-sem 'without evaluations' > enrolled (source data quirk): 3


## 5. Duplicate records
There is no student ID column, so we can only check for rows that are identical across every column.

In [7]:
print("Fully duplicated rows:", int(df.duplicated().sum()))

Fully duplicated rows: 0


## 6. Data types
Almost every column is an **integer code** (a category stored as a number), not a real quantity. For example, `Course = 9500` means *Nursing*, not the number 9,500.

In [8]:
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "unique_values": df.nunique(),
    "min": df.min(),
    "max": df.max(),
})
summary

,dtype,unique_values,min,max
Marital status,int64,6,1,6
Application mode,int64,18,1,57
Application order,int64,8,0,9
Course,int64,17,33,9991
Daytime/evening attendance\t,int64,2,0,1
Previous qualification,int64,17,1,43
Previous qualification (grade),float64,101,95.0,190.0
Nacionality,int64,21,1,109
Mother's qualification,int64,29,1,44
Father's qualification,int64,34,1,44


## 7. Descriptive statistics

In [9]:
df.describe().T.round(2)

,count,mean,std,min,25%,50%,75%,max
Marital status,4424.0,1.18,0.61,1.00,1.00,1.00,1.00,6.00
Application mode,4424.0,18.67,17.48,1.00,1.00,17.00,39.00,57.00
Application order,4424.0,1.73,1.31,0.00,1.00,1.00,2.00,9.00
Course,4424.0,8856.64,2063.57,33.00,9085.00,9238.00,9556.00,9991.00
Daytime/evening attendance\t,4424.0,0.89,0.31,0.00,1.00,1.00,1.00,1.00
Previous qualification,4424.0,4.58,10.22,1.00,1.00,1.00,1.00,43.00
Previous qualification (grade),4424.0,132.61,13.19,95.00,125.00,133.10,140.00,190.00
Nacionality,4424.0,1.87,6.91,1.00,1.00,1.00,1.00,109.00
Mother's qualification,4424.0,19.56,15.60,1.00,2.00,19.00,37.00,44.00
Father's qualification,4424.0,22.28,15.34,1.00,3.00,19.00,37.00,44.00


### Outcome variable (`Target`)

In [10]:
t = df["Target"].value_counts()
pd.DataFrame({"students": t, "percent": (t / len(df) * 100).round(1)})

,students,percent
Target,,
Graduate,2209,49.9
Dropout,1421,32.1
Enrolled,794,17.9


## 8. What each feature represents
Code meanings come from the UCI documentation. The lookup tables below are used later to create readable labels in the **processed** dataset.

In [11]:
COURSE = {
    33: "Biofuel Production Technologies", 171: "Animation and Multimedia Design",
    8014: "Social Service (evening)", 9003: "Agronomy", 9070: "Communication Design",
    9085: "Veterinary Nursing", 9119: "Informatics Engineering", 9130: "Equinculture",
    9147: "Management", 9238: "Social Service", 9254: "Tourism", 9500: "Nursing",
    9556: "Oral Hygiene", 9670: "Advertising and Marketing Management",
    9773: "Journalism and Communication", 9853: "Basic Education", 9991: "Management (evening)",
}
NATIONALITY = {
    1: "Portuguese", 2: "German", 6: "Spanish", 11: "Italian", 13: "Dutch", 14: "English",
    17: "Lithuanian", 21: "Angolan", 22: "Cape Verdean", 24: "Guinean", 25: "Mozambican",
    26: "Santomean", 32: "Turkish", 41: "Brazilian", 62: "Romanian", 100: "Moldovan",
    101: "Mexican", 103: "Ukrainian", 105: "Russian", 108: "Cuban", 109: "Colombian",
}
GENDER = {1: "Male", 0: "Female"}
ATTENDANCE = {1: "Daytime", 0: "Evening"}
MARITAL = {1: "Single", 2: "Married", 3: "Widower", 4: "Divorced", 5: "Facto union", 6: "Legally separated"}

print("Programmes (Course):")
print(df["Course"].map(COURSE).value_counts().to_string())

Programmes (Course):
Course
Nursing                                 766
Management                              380
Social Service                          355
Veterinary Nursing                      337
Journalism and Communication            331
Advertising and Marketing Management    268
Management (evening)                    268
Tourism                                 252
Communication Design                    226
Animation and Multimedia Design         215
Social Service (evening)                215
Agronomy                                210
Basic Education                         192
Informatics Engineering                 170
Equinculture                            141
Oral Hygiene                             86
Biofuel Production Technologies          12


In [12]:
print("Gender:\n", df["Gender"].map(GENDER).value_counts().to_string(), "\n")
print("Attendance:\n", df["Daytime/evening attendance\t"].map(ATTENDANCE).value_counts().to_string(), "\n")
print("Top nationalities:\n", df["Nacionality"].map(NATIONALITY).value_counts().head(8).to_string())

Gender:
 Gender
Female    2868
Male      1556 

Attendance:
 Daytime/evening attendance\t
Daytime    3941
Evening     483 

Top nationalities:
 Nacionality
Portuguese      4314
Brazilian         38
Santomean         14
Spanish           13
Cape Verdean      13
Guinean            5
Moldovan           3
Italian            3


### Feature dictionary (plain language)

| Group | Column(s) | Meaning |
|---|---|---|
| Demographics | `Marital status`, `Gender` (1 = male, 0 = female), `Age at enrollment`, `Nacionality`, `International` (1 = yes) | Who the student is |
| Admission | `Application mode`, `Application order` (0 = first choice … 9 = last), `Previous qualification`, `Previous qualification (grade)` (0–200), `Admission grade` (0–200) | How and with what grades the student entered |
| Programme | `Course` (17 undergraduate degree programmes), `Daytime/evening attendance` (1 = daytime, 0 = evening) | What and when they study. **Evening is not the same as online.** |
| Family background | `Mother's/Father's qualification`, `Mother's/Father's occupation` | Parents' education and job codes |
| Socio-economic | `Displaced`, `Educational special needs`, `Debtor`, `Tuition fees up to date`, `Scholarship holder` (all 1 = yes) | Financial and support status |
| 1st-semester performance | `Curricular units 1st sem (credited / enrolled / evaluations / approved / grade / without evaluations)` | Course units taken, sat and passed in the **first semester of year 1**; grade is the average of **passed units only** on the Portuguese **0–20** scale (10 = pass). It is 0 when nothing was passed, so failures never appear in it |
| 2nd-semester performance | same six columns for the 2nd semester | Same, for the **second semester of year 1** |
| Macro-economy | `Unemployment rate`, `Inflation rate`, `GDP` | National economic indicators at enrolment time. Only about 10 distinct values, one per entry year, **but the year itself is not given** |
| Outcome | `Target` | Status at the end of the programme's normal duration: **Dropout**, **Enrolled** (still studying) or **Graduate** |

## 9. Which features support the dashboard?

Summary only. The full gap analysis is in the project `README.md`.

| Dashboard module | Supported by the UCI data | Not in the UCI data |
|---|---|---|
| **A. Enrollment** | per programme (`Course`), gender, nationality, international flag, daytime vs evening | calendar semester/year, school/faculty, online vs face-to-face |
| **B. Academic performance** | 1st- and 2nd-semester average grade (GPA proxy), units approved/enrolled ratio, so poor standing can be *derived* | dates, semesters after year 1, official warning/probation records |
| **C. Retention** | `Target` gives a final outcome per student | any year-to-year enrolment records, so Fall→Fall, Spring→Spring and Summer→Summer **cannot** be computed |
| **D. Graduation** | `Target = Graduate` for undergraduates | graduation date, time-to-degree, Master's and Doctorate students |

In [13]:
# Derived indicators, computed in memory only (the raw file is not modified)
derived = pd.DataFrame({
    "programme": df["Course"].map(COURSE),
    "sem1_gpa_0_20": df[g1],
    "sem2_gpa_0_20": df[g2],
    "sem1_pass_ratio": (df["Curricular units 1st sem (approved)"] / df[e1]).where(df[e1] > 0),
    "sem2_pass_ratio": (df["Curricular units 2nd sem (approved)"] / df[e2]).where(df[e2] > 0),
    "target": df["Target"],
})
# Example rule only; the real academic-standing rule must be agreed before use
derived["example_poor_standing_sem1"] = (derived["sem1_pass_ratio"] < 0.5)
print("Example: share of students passing < 50% of 1st-sem units:",
      round(derived["example_poor_standing_sem1"].mean() * 100, 1), "%")
derived.groupby("target")[["sem1_gpa_0_20", "sem2_gpa_0_20", "sem1_pass_ratio"]].mean().round(2)

Example: share of students passing < 50% of 1st-sem units: 19.2 %


,sem1_gpa_0_20,sem2_gpa_0_20,sem1_pass_ratio
target,,,
Dropout,7.26,5.90,0.41
Enrolled,11.13,11.12,0.71
Graduate,12.64,12.70,0.93


In [14]:
# Outcome by programme (the closest thing to graduation/dropout rates the real data supports)
pd.crosstab(derived["programme"], derived["target"], normalize="index").mul(100).round(1).sort_values("Graduate")

target,Dropout,Enrolled,Graduate
programme,,,
Informatics Engineering,54.1,37.6,8.2
Biofuel Production Technologies,66.7,25.0,8.3
Management (evening),50.7,20.1,29.1
Basic Education,44.3,26.0,29.7
Equinculture,55.3,14.9,29.8
Management,35.3,28.4,36.3
Agronomy,41.0,17.6,41.4
Oral Hygiene,38.4,19.8,41.9
Animation and Multimedia Design,38.1,17.2,44.7


## 10. Conclusions

1. The file matches the UCI dataset exactly: 4,424 students, 37 columns, no missing cells and no duplicate rows.
2. **There is no student ID and no date column.** Each row is one student, measured only in their first year.
3. Every student is an **undergraduate** at one institution. There are no Master's or Doctorate records.
4. Modules A and B can be partly built from real data. Calendar-based retention (C) and time-based graduation rates (D) **need synthetic data**, and that synthetic data must always be labelled as synthetic.
5. Next step: agree the database schema, then build `data/processed/` (cleaned column names and labels, plus a `record_source = 'UCI'` flag and a generated `student_id`).